<a href="https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter06/06_03_oblique_shock_collision_slip_line.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Head-On Collision of Oblique Shocks and Slip-Line Matching

**Book:** *Introduction to Compressible Flows*<br>
**Chapter 6:** Oblique Shock Waves and Expansion Waves<br>
**Concepts:** interacting oblique shocks · pressure compatibility · slip-line angle · root finding

## Learning objectives

- Propagate the upper and lower incident shocks through two downstream states.
- Enforce equal static pressure across the slip line.
- Compare a bracketed hybrid root solve with a deliberately slow reference iteration.

## How to use this notebook

Run the fast solver cell for the deterministic example. The final cell retains the slow reference algorithm as an optional function and does not run automatically.

> **Model scope.** The construction assumes attached shocks, a perfect gas, and an inviscid slip line; inputs outside the attached-shock domain can be nonphysical.


In [ ]:
import math
from typing import Tuple


def oblique_shock(M1: float, P1: float, theta: float, gamma: float = 1.4) -> Tuple[float, float]:

    tan_theta = math.tan(theta)

    landa = math.sqrt(
        (M1**2 - 1)**2
        - 3.0 * (1.0 + (gamma - 1.0) / 2.0 * M1**2)
        * (1.0 + (gamma + 1.0) / 2.0 * M1**2)
        * (tan_theta**2)
    )

    X_num = (
        (M1**2 - 1)**3
        - 9.0 * (1.0 + (gamma - 1.0) / 2.0 * M1**2)
        * (1.0 + (gamma - 1.0) / 2.0 * M1**2 + (gamma + 1.0) / 4.0 * M1**4)
        * (tan_theta**2)
    )
    X = X_num / (landa**3)

    # Clamp to avoid acos domain errors
    X = max(-1.0, min(1.0, X))

    beta_tan = (
        (M1**2 - 1.0 + 2.0 * landa * math.cos((4.0 * math.pi + math.acos(X)) / 3.0))
        / (3.0 * (1.0 + (gamma - 1.0) / 2.0 * M1**2) * tan_theta)
    )
    beta = math.atan(beta_tan)

    M1N = M1 * math.sin(beta)

    M2N = math.sqrt(
        M1N**2
        * (
            ((gamma - 1.0) / (gamma + 1.0) + 2.0 / ((gamma + 1.0) * M1N**2))
            / (1.0 + 2.0 * gamma / (gamma + 1.0) * (M1N**2 - 1.0))
        )
    )
    M2 = M2N / math.sin(beta - theta)

    pressureratio = 1.0 + (2.0 * gamma) / (gamma + 1.0) * (M1N**2 - 1.0)
    P2 = P1 * pressureratio
    return M2, P2


def solve_slipline_fast(
    M1: float,
    theta2: float,
    theta3: float,
    P1: float = 1.0,
    tol_abs: float = 1e-3,
    max_iter: int = 80,
) -> dict:
    """
    Fast solver that finds gsld such that P4 ~= P5.
    Returns a dict of results including angles, Mach numbers, pressures, and slip line angle.
    """

    # First shocks
    M2, P2 = oblique_shock(M1, P1, theta2)
    M3, P3 = oblique_shock(M1, P1, theta3)

    # Determine which branch should be used (based on initial P2,P3)
    p3_gt_p2 = (P3 > P2)

    # Build function f(gsld) = P4 - P5
    def eval_f(gsld: float) -> Tuple[float, float, float, float, float, float]:
        if p3_gt_p2:
            theta4 = theta2 + gsld
            theta5 = theta3 - gsld
        else:
            theta4 = theta2 - gsld
            theta5 = theta3 + gsld

        M4, P4 = oblique_shock(M2, P2, theta4)
        M5, P5 = oblique_shock(M3, P3, theta5)
        return (P4 - P5, theta4, theta5, M4, P4, P5)

    # --- Bracket the root ---
    gsld0 = abs(theta3 - theta2)
    f0, *_ = eval_f(gsld0)

    # We want to find a and b with f(a)*f(b) <= 0
    # Expand range geometrically until we bracket or give up.
    a = max(0.0, 0.0)  # allow 0
    fa, *_ = eval_f(a)

    b = max(gsld0, 1e-8)
    fb, *_ = eval_f(b)

    if fa == 0.0:
        # already solved at gsld=0
        root = a
        f_root, theta4, theta5, M4, P4, P5 = eval_f(root)
    else:
        # If not bracketed, expand b
        expand = 1.6
        tries = 0
        while fa * fb > 0 and tries < 60:
            b *= expand
            fb, *_ = eval_f(b)
            tries += 1

        if fa * fb > 0:
            # Could not bracket; fallback to a damped secant around gsld0
            x0 = gsld0
            x1 = gsld0 * 1.05 + 1e-8
            f_x0, *_ = eval_f(x0)
            f_x1, *_ = eval_f(x1)

            for _ in range(max_iter):
                if abs(f_x1) < tol_abs:
                    root = x1
                    break
                denom = (f_x1 - f_x0)
                if denom == 0:
                    x1 = x1 * 0.9
                    f_x1, *_ = eval_f(x1)
                    continue
                x2 = x1 - f_x1 * (x1 - x0) / denom
                # damping to avoid wild jumps
                x2 = 0.7 * x2 + 0.3 * x1
                if x2 < 0:
                    x2 = 0.0
                x0, f_x0 = x1, f_x1
                x1 = x2
                f_x1, *_ = eval_f(x1)
            else:
                root = x1

            f_root, theta4, theta5, M4, P4, P5 = eval_f(root)
        else:
            # --- Root solve with robust bisection + secant (Illinois-ish hybrid) ---
            left, right = a, b
            fL, fR = fa, fb

            x = None
            fx = None
            theta4 = theta5 = M4 = P4 = P5 = float("nan")

            for _ in range(max_iter):
                # Secant step if safe, else bisection
                if (fR - fL) != 0:
                    x_sec = right - fR * (right - left) / (fR - fL)
                else:
                    x_sec = (left + right) / 2.0

                # Keep it inside bracket
                if not (left < x_sec < right):
                    x_sec = (left + right) / 2.0

                fx, theta4, theta5, M4, P4, P5 = eval_f(x_sec)

                if abs(fx) < tol_abs:
                    x = x_sec
                    break

                # Update bracket
                if fL * fx <= 0:
                    right, fR = x_sec, fx
                else:
                    left, fL = x_sec, fx

                # If bracket is tiny, stop
                if abs(right - left) < 1e-12:
                    x = (left + right) / 2.0
                    fx, theta4, theta5, M4, P4, P5 = eval_f(x)
                    break

            if x is None:
                x = (left + right) / 2.0
                fx, theta4, theta5, M4, P4, P5 = eval_f(x)

            root = x
            f_root = fx

    # Slip line angle
    if theta2 < theta3:
        slip_line_angle = root
    else:
        slip_line_angle = -root

    return {
        "M1": M1,
        "P1": P1,
        "theta2_rad": theta2,
        "theta3_rad": theta3,
        "M2": M2,
        "P2": P2,
        "M3": M3,
        "P3": P3,
        "gsld": root,
        "theta4_rad": theta4,
        "theta5_rad": theta5,
        "M4": M4,
        "P4": P4,
        "P5": P5,
        "P4_minus_P5": f_root,
        "slip_line_angle_rad": slip_line_angle,
        "slip_line_angle_deg": slip_line_angle * 180.0 / math.pi,
    }


def main():
    print("Please Enter Mach Number of Flow and press Enter")
    M1 = float(input().strip())

    print("Please Enter Nuzzle upper degree and press Enter")
    theta2_deg = float(input().strip())

    print("Please Enter Nuzzle lower degree and press Enter")
    theta3_deg = float(input().strip())

    theta2 = theta2_deg * math.pi / 180.0
    theta3 = theta3_deg * math.pi / 180.0

    res = solve_slipline_fast(M1, theta2, theta3, P1=1.0, tol_abs=1e-3, max_iter=80)

    print("================================================================================")
    print("Converged (fast solver)")
    print(f"P4={res['P4']:.6f}  P5={res['P5']:.6f}  |P4-P5|={abs(res['P4_minus_P5']):.6f}")
    print(f"gsld(rad)={res['gsld']:.8e}")
    print(f"Slip Line Angle (deg)={res['slip_line_angle_deg']:.6f}")
    print("theta4(deg)=", res["theta4_rad"] * 180.0 / math.pi)
    print("theta5(deg)=", res["theta5_rad"] * 180.0 / math.pi)
    print("================================================================================")






# Reproducible worked example
example = solve_slipline_fast(
    M1=3.0,
    theta2=math.radians(8.0),
    theta3=math.radians(12.0),
    P1=1.0,
)
print(f"P4 = {example['P4']:.6f}, P5 = {example['P5']:.6f}")
print(f"|P4-P5| = {abs(example['P4_minus_P5']):.3e}")
print(f"slip-line angle = {example['slip_line_angle_deg']:.6f} deg")


In [ ]:
import math
def oblique_shock(M1: float, P1: float, theta: float, gamma: float = 1.4):


    # --- Calculation of beta (shock angle) ---
    tan_theta = math.tan(theta)

    landa = math.sqrt(
        (M1**2 - 1)**2
        - 3 * (1 + (gamma - 1) / 2 * M1**2) * (1 + (gamma + 1) / 2 * M1**2) * (tan_theta**2)
    )

    X_num = (
        (M1**2 - 1)**3
        - 9 * (1 + (gamma - 1) / 2 * M1**2)
        * (1 + (gamma - 1) / 2 * M1**2 + (gamma + 1) / 4 * M1**4)
        * (tan_theta**2)
    )
    X_den = landa**3
    X = X_num / X_den

    # Guard against tiny numerical overshoot beyond [-1, 1]
    X = max(-1.0, min(1.0, X))

    bethaTang = (
        (M1**2 - 1 + 2 * landa * math.cos((4 * math.pi + math.acos(X)) / 3))
        / (3 * (1 + (gamma - 1) / 2 * M1**2) * tan_theta)
    )
    beta = math.atan(bethaTang)

    # --- Pressure ratio and other properties ---
    M1N = M1 * math.sin(beta)


    M2N = math.sqrt(
        M1N**2
        * (
            ((gamma - 1) / (gamma + 1) + 2 / ((gamma + 1) * M1N**2))
            / (1 + 2 * gamma / (gamma + 1) * (M1N**2 - 1))
        )
    )

    M2 = M2N / math.sin(beta - theta)

    densityratio = ((gamma + 1) * M1N**2) / (2 + (gamma - 1) * M1N**2)
    pressureratio = 1 + (2 * gamma) / (gamma + 1) * (M1N**2 - 1)

    P2 = P1 * pressureratio
    return M2, P2


def slow_reference_demo():
    print("Please Enter Mach Number of Flow and press Enter")
    M1 = float(input().strip())

    print("Please Enter Nuzzle upper degree and press Enter")
    theta2_deg = float(input().strip())

    print("Please Enter Nuzzle lower degree and press Enter")
    theta3_deg = float(input().strip())

    theta2 = theta2_deg * math.pi / 180.0
    theta3 = theta3_deg * math.pi / 180.0

    P1 = 1.0
    epsilon = 1e-4

    M2, P2 = oblique_shock(M1, P1, theta2)
    M3, P3 = oblique_shock(M1, P1, theta3)

    # Initial guess for gsld and (theta4, theta5)
    if P3 > P2:
        gsld = theta3 - theta2
        theta4 = theta2 + gsld
        theta5 = theta3 - gsld
    else:
        gsld = theta2 - theta3
        theta4 = theta2 - gsld
        theta5 = theta3 + gsld

    M4, P4 = oblique_shock(M2, P2, theta4)
    M5, P5 = oblique_shock(M3, P3, theta5)

    for i in range(1, 200000 + 1):

        if (P3 > P2) and (P5 > P4):
            gsld += epsilon
            theta4 = theta2 + gsld
            theta5 = theta3 - gsld

        if (P3 > P2) and (P4 > P5):
            gsld -= epsilon
            theta4 = theta2 + gsld
            theta5 = theta3 - gsld

        if (P2 > P3) and (P5 > P4):
            gsld -= epsilon
            theta4 = theta2 - gsld
            theta5 = theta3 + gsld

        if (P2 > P3) and (P4 > P5):
            gsld += epsilon
            theta4 = theta2 - gsld
            theta5 = theta3 + gsld

        M4, P4 = oblique_shock(M2, P2, theta4)
        M5, P5 = oblique_shock(M3, P3, theta5)

        print("================================================================================")
        print("Iteration Number", i)
        print("P4=", P4, "AND P5=", P5)

        if abs(P4 - P5) < 0.001:
            print("converged")
            break

        if theta2 < theta3:
            slip_line_angle = gsld
        else:
            slip_line_angle = -1.0 * gsld

        print("Slip Line Angle=", slip_line_angle * 180.0 / math.pi)
        print("================================================================================")



    input("Press Enter to exit...")






# Optional: call slow_reference_demo() to reproduce the original incremental iteration.


## Interpretation and verification checklist

- The converged pressure mismatch |P4-P5| should satisfy the declared tolerance.
- Verify that every intermediate normal Mach number is supersonic before applying an oblique-shock relation.
- Report both the sign convention and magnitude of the slip-line angle.

## Reproducibility note

Restart the runtime and run the notebook from top to bottom after changing inputs. Record the input values, units, heat-capacity ratio, numerical tolerances, and dependency versions with any result used in coursework, research, or a publication.

Questions or reproducibility reports are welcome through the repository's issue templates.
